### RAG Pipeline - Data Ingestion to VectorDB pipeline

In [9]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\kaush\AppData\Local\Temp\ipykernel_25488\2096081597.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [10]:
### Read all the pdf inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: 1. exam_dwm_report.pdf
  ✓ Loaded 14 pages

Processing: Model_Performance__Audit_System.pdf
  ✓ Loaded 12 pages

Processing: OS Lab Report B240045CS.pdf
  ✓ Loaded 14 pages

Processing: Skin Cancer Detection Report.pdf
  ✓ Loaded 6 pages

Total documents loaded: 46


In [11]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'exam_dwm_report', 'source': '..\\data\\pdf\\1. exam_dwm_report.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': '1. exam_dwm_report.pdf', 'file_type': 'pdf'}, page_content='NATIONAL  INSTITUTE  OF  TECHNOLOGY  SIKKIM  \n \n  DATA  WAREHOUSING  AND  MINING  LABORATORY  \n(AI14201/CS14211)\n  Project  Report  on  SKIN  CANCER  DETECTION   SUBMITTED  BY:  Naveen  Verma  (B240014AI)  Nitin  Kumar  Jha  (B240016AI)  Kausal  Sharma  (B240045CS)  Pawan  Kumar  Limboo  (B240055CS)   SUBMITTED  TO:  Dr.  Krishna  Kumar  (Assistant  Professor)   DEPARTMENT  OF  COMPUTER  SCIENCE  AND  \nENGINEERING'),
 Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'exam_dwm_report', 'source': '..\\data\\pdf\\1. exam_dwm_report.pdf', 'total_pages': 14, 'page': 1, 'page_label': '2', 'source_file': '1. exa

In [12]:
### Text Splitting into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk: ")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [13]:
chunks=split_documents(all_pdf_documents)
chunks

Split 46 documents into 102 chunks

Example chunk: 
Content: NATIONAL  INSTITUTE  OF  TECHNOLOGY  SIKKIM  
 
  DATA  WAREHOUSING  AND  MINING  LABORATORY  
(AI14201/CS14211)
  Project  Report  on  SKIN  CANCER  DETECTION   SUBMITTED  BY:  Naveen  Verma  (B24001...
Metadata: {'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'exam_dwm_report', 'source': '..\\data\\pdf\\1. exam_dwm_report.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': '1. exam_dwm_report.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'exam_dwm_report', 'source': '..\\data\\pdf\\1. exam_dwm_report.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': '1. exam_dwm_report.pdf', 'file_type': 'pdf'}, page_content='NATIONAL  INSTITUTE  OF  TECHNOLOGY  SIKKIM  \n \n  DATA  WAREHOUSING  AND  MINING  LABORATORY  \n(AI14201/CS14211)\n  Project  Report  on  SKIN  CANCER  DETECTION   SUBMITTED  BY:  Naveen  Verma  (B240014AI)  Nitin  Kumar  Jha  (B240016AI)  Kausal  Sharma  (B240045CS)  Pawan  Kumar  Limboo  (B240055CS)   SUBMITTED  TO:  Dr.  Krishna  Kumar  (Assistant  Professor)   DEPARTMENT  OF  COMPUTER  SCIENCE  AND  \nENGINEERING'),
 Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'exam_dwm_report', 'source': '..\\data\\pdf\\1. exam_dwm_report.pdf', 'total_pages': 14, 'page': 1, 'page_label': '2', 'source_file': '1. exa

### Embedding and Vector DB

In [14]:
from dotenv import load_dotenv
import os

load_dotenv()

hf_token = os.getenv("HF_TOKEN")
print(hf_token is not None)

True


In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    token=hf_token
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [18]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: sentence-transformers/multi-qa-MiniLM-L6-cos-v1


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


C:\Users\kaush\AppData\Local\Temp\ipykernel_25488\2885700169.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [19]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [20]:
chunks

[Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'exam_dwm_report', 'source': '..\\data\\pdf\\1. exam_dwm_report.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': '1. exam_dwm_report.pdf', 'file_type': 'pdf'}, page_content='NATIONAL  INSTITUTE  OF  TECHNOLOGY  SIKKIM  \n \n  DATA  WAREHOUSING  AND  MINING  LABORATORY  \n(AI14201/CS14211)\n  Project  Report  on  SKIN  CANCER  DETECTION   SUBMITTED  BY:  Naveen  Verma  (B240014AI)  Nitin  Kumar  Jha  (B240016AI)  Kausal  Sharma  (B240045CS)  Pawan  Kumar  Limboo  (B240055CS)   SUBMITTED  TO:  Dr.  Krishna  Kumar  (Assistant  Professor)   DEPARTMENT  OF  COMPUTER  SCIENCE  AND  \nENGINEERING'),
 Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'exam_dwm_report', 'source': '..\\data\\pdf\\1. exam_dwm_report.pdf', 'total_pages': 14, 'page': 1, 'page_label': '2', 'source_file': '1. exa

In [21]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store in the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 102 texts...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Generated embeddings with shape: (102, 384)
Adding 102 documents to vector store...
Successfully added 102 documents to vector store
Total documents in collection: 102


### Retriever Pipeline from VectorStore

In [49]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [23]:
rag_retriever

In [26]:
rag_retriever.retrieve("What is skin cancer?")

Retrieving documents for query: 'What is skin cancer?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_fce2e3b3_2',
  'content': 'being\n \nstored\n \non\n \nthe\n \ncloud\n \nvia\n \nthe\n \nSupabase\n \n(PostgreSQL)\n \nrelational\n \ndata\n \nwarehouse.\n \nA\n \nCNN\n \nis\n \nused\n \nto\n \nautomatically\n \nclassify\n \nmalignant/benign\n \nskin\n \nlesions\n \n(MEL,\n \nBCC,\n \nand\n \nSCC).\n \nThe\n \npipeline\n \nis\n \nfully\n \noperational\n \nvia\n \na\n \nRESTful\n \nAPI\n \nand\n \nhosted\n \non\n \na\n \nRender\n \ncloud\n \nserver\n \nto\n \nprovide\n \nimmediate,\n \nlow-latency\n \nresults\n \nto\n \na\n \nclinical\n \ndashboard\n \nhosted\n \non\n \na\n \nweb\n \napplication.\n \n \n1.  Introduction  \n1.1.  Background  on  Oncological  Dermatology  \nSkin  cancer  is  one  of  the  most  common  and  fast-growing  cancers  in  the  world.  Pathologically  \nthe\n \nmost\n \nfrequent\n \nskin\n \ncancers\n \nare\n \nMelanoma\n \n(MEL),\n \nwhich\n \nis\n \nan\n \naggressive\n \ncancer\n \ncausing\n \nmost\n \nskin\n \ncancer\n \ndeaths,\n \nBasal\n \nC

In [50]:
rag_retriever.retrieve("What is automated data mining?")

Retrieving documents for query: 'What is automated data mining?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_91c00c15_10',
  'content': 'automatically\n \n(mine)\n \nweb-scale\n \nunstructured\n \nskin\n \nlesion\n \nimages\n \nfrom\n \npublic\n \nclinical\n \nspaces,\n \nprocess\n \nthem\n \nand\n \ningested\n \ninto\n \nan\n \nstructured\n \ncloud\n \nbased\n \ndata-warehouse\n \nand\n \nrun\n \na\n \ndeep\n \nconvolutional\n \nneural\n \nnetwork\n \nto\n \nprovide\n \non-demand\n \ndiagnostic\n \nclass\n \n(MEL,\n \nBCC,\n \nSCC)\n \ninformation\n \nthrough\n \nan\n \ninteractive,\n \nmedical-grade\n \npresentation\n \nlayer.\n \n \n4.  Objectives  \nIn  order  to  address  the  central  problem  statement,  the  project  implements  ﬁve  essential  \ntechnical\n \nmilestones:\n 1.  Automated  Data  Mining  -  Create  a  tailored  web-scraping  engine  to  automatically  \nparse\n \nand\n \nstore\n \nmedical\n \nimages,\n \nand\n \nthe\n \nrelated\n \nmedical\n \nimaging\n \nmetadata,\n \nfrom\n \npublic\n \nclinical\n \nnetworks.\n 2.  ETL  Pipeline  Tuning  -  Build  an  opt

### RAG Pipeline- VectorDB To LLM Output Generation

In [31]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

In [34]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [35]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.
Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}
Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [36]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [38]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Unified Multi-task Learning Framework",score_threshold=-999)

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: -999
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_f935aa15_13',
  'content': '5.1.  Architectural  Breakdown  \nThe  framework  employs  a  distributed  multi-tier  client-server  architecture.  The  presentation  \ntier\n \nreceives\n \nthe\n \nusers\n \nparameters,\n \nwhether\n \nthat\n \nis\n \na\n \nURL\n \nstring\n \nfor\n \nweb\n \nscale\n \nharvesting\n \nor\n \nlocalized\n \nclinical\n \nimagery\n \nfor\n \ndiagnostic\n \napplication.\n \nThese\n \nasynchronized\n \npayloads\n \nare\n \nsent\n \nto\n \na\n \ncentral\n \nPython\n \nFlask\n \nmiddleware\n \nover\n \nHTTP\n \nPOST\n \nmethods,\n \nwhere\n \nthe\n \nbackend\n \nis\n \nsplit\n \ninto\n \ntwo\n \nseparate\n \nprocessing\n \ntiers,\n \nan\n \nExtraction\n \nETL\n \nEngine\n \nusing\n \nBeautifulSoup4\n \nand\n \nRequests\n \nto\n \nprocess\n \nexternal\n \nweb\n \nresources,\n \nand\n \nan\n \nautomatic\n \nMachine\n \nLearning\n \nInference\n \nEngine\n \nusing\n \nTensorFlow,\n \nKeras,\n \nand\n \nOpenCV,\n \nto\n \nperform\n \nthe\n \ndiagnostics\n 

### Integration Vectordb Context pipeline With LLM output

In [43]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [44]:
answer=rag_simple("What is Skin Cancer?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Skin Cancer?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Supervisor: Dr. S. S. Iyengar
Institution: IIT Bombay
Project Title: Skin Cancer Detection
Project Description: The project aims to develop a deep learning-based skin cancer detection system using a Convolutional Neural Network (CNN) to classify skin lesions into malignant and benign categories. The system will be integrated with a clinical dashboard to provide immediate and low-latency results to clinicians.

The project aims to develop a deep learning-based skin cancer detection system using a Convolutional Neural Network (CNN) to classify skin lesions into malignant and benign categories. The system will be integrated with a clinical dashboard to provide immediate and low-latency results to clinicians.

The project aims to develop a deep learning-based skin cancer detection system using a Convolutional Neural Network (CNN) to classify skin lesions into malignant and benign categories. The system will b

### Enhanced RAG Pipeline Features

In [51]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is data warehousing?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is data warehousing?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Answer:  and type.
Sources: [{'source': 'Skin Cancer Detection Report.pdf', 'page': 3, 'score': 0.15724033117294312, 'preview': 'Dashboard Home, 2. Live Scraper Logs, and 3. Prediction Result here).\n12.3 Dataset Distribution\nThrough the DWM pipeline, the system tracks the origin of all data, rendering real-time statistics that\nshow the breakdown of the warehouse by class \x00MEL vs. BCC) and by source domain.\n\x00\x00\x00\x00Advantages\nAuto...'}]
Confidence: 0.15724033117294312
Context Preview: Dashboard Home, 2. Live Scraper Logs, and 3. Prediction Result here).
12.3 Dataset Distribution
Through the DWM pipeline, the system tracks the origin of all data, rendering real-time statistics that
show the breakdown of the warehouse by class  MEL vs. BCC) and by source domain.
    Advantages
Auto


In [53]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content
        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is the key difference between windows and linux?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What is the key difference between windows and linux?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Comparison Between Linux and 
Windows Operating Systems 
 
⮚ Introduction: 
Two of the most widely used operating systems today are Linux 
and Microsoft Windows which differ significantly in architecture 
and usage . 
 
Linux is an open -source, Unix-like operating system originally 
developed by Linux Torvalds in 1991.Linux was origin ally 
designed as a clone of UNIX and is released under the copyleft 
HPL license. Linux was developed following the lack of a 
working kernel for GNU,a UNIX -compatible operating system 
made entirely of free software that had been in develompent since 
1993 by the GNU Project. [12] 
 
Microsoft Windows is a graphical operating system developed 
and marketed by Microsoft. It is the most popular desktop 
operating system in the world. It is grouped into families – 
Windows for personal co